# LeetCode 30 — Substring with Concatenation of All Words (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findSubstring(self, s: str, words: List[str]) -> List[int]:

```

Đây là một bài **Hard** nhưng ý tưởng lõi lại rất "đời thường": **đếm từ** và **trượt cửa sổ**.

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì, "chuỗi ghép" (concatenated string) nghĩa là gì.
2. Cách cắt chuỗi bằng slicing `s[i:j]` và `range(start, stop, step)` để chia chuỗi thành từng **từ** cùng độ dài.
3. Cách **đếm tần suất** bằng `dict` và `collections.Counter`.
4. Ba cách giải đi từ dễ đến tối ưu:
   - **Cách 1:** Brute force sinh mọi hoán vị (chỉ để hiểu đề và thấy nó "nổ" thế nào).
   - **Cách 2:** Với mỗi vị trí bắt đầu, cắt thành từng từ rồi so sánh `Counter`.
   - **Cách 3 (cách chính):** **Sliding window** (cửa sổ trượt) theo từng offset `0..word_len-1`, độ phức tạp `O(n · word_len)`.
5. Vì sao cách 3 đúng, lỗi hay gặp, cách nhận dạng pattern và các bài liên quan.

## 1. Đề bài (theo LeetCode)

You are given a string `s` and an array of strings `words`. All the strings of `words` are of **the same length**.

A **concatenated string** is a string that exactly contains all the strings of any permutation of `words` concatenated.

- For example, if `words = ["ab","cd","ef"]`, then `"abcdef"`, `"abefcd"`, `"cdabef"`, `"cdefab"`, `"efabcd"`, and `"efcdab"` are all concatenated strings. `"acdbef"` is not a concatenated string because it is not the concatenation of any permutation of `words`.

Return an array of *the starting indices of all the concatenated substrings in* `s`. You can return the answer in **any order**.

### Dịch nhanh

- Cho chuỗi `s` và danh sách `words`, **mọi từ trong `words` dài bằng nhau**.
- Một **chuỗi ghép** là chuỗi tạo ra bằng cách nối **tất cả** các từ trong `words` theo **một thứ tự nào đó** (một **hoán vị** — permutation — tức là một cách sắp xếp lại thứ tự).
- Mỗi từ phải dùng **đúng số lần nó xuất hiện** trong `words` (nếu `words` có 2 chữ `"word"` thì chuỗi ghép phải chứa đúng 2 lần `"word"`).
- Trả về **mọi chỉ số bắt đầu** trong `s` mà tại đó có một chuỗi con là chuỗi ghép. Thứ tự kết quả không quan trọng.



### Ví dụ trong đề

**Example 1**

```text
Input:  s = "barfoothefoobarman", words = ["foo","bar"]
Output: [0,9]
```

- Chuỗi con bắt đầu tại `0` là `"barfoo"` = `"bar"` + `"foo"`.
- Chuỗi con bắt đầu tại `9` là `"foobar"` = `"foo"` + `"bar"`.
- Trả `[9,0]` cũng được.

**Example 2**

```text
Input:  s = "wordgoodgoodgoodbestword", words = ["word","good","best","word"]
Output: []
```

- Không có chuỗi con nào chứa **đúng 2 lần** `"word"`, 1 lần `"good"`, 1 lần `"best"` liền nhau.

**Example 3**

```text
Input:  s = "barfoofoobarthefoobarman", words = ["bar","foo","the"]
Output: [6,9,12]
```

- Tại `6`: `"foobarthe"`; tại `9`: `"barthefoo"`; tại `12`: `"thefoobar"`.

### Constraints

- `1 <= s.length <= 10^4`
- `1 <= words.length <= 5000`
- `1 <= words[i].length <= 30`
- `s` and `words[i]` consist of lowercase English letters.

## 2. Hiểu đề bằng tay: chuỗi con cần dài bao nhiêu?

Đặt tên cho các đại lượng (sẽ dùng suốt notebook):

| Ký hiệu | Ý nghĩa | Example 1 |
|---|---|---|
| `n` | độ dài `s` | `18` |
| `m` | số từ trong `words` | `2` |
| `k` | độ dài **một** từ (`word_len`) | `3` |
| `total = m * k` | độ dài chuỗi ghép | `6` |

Vì chuỗi ghép chứa **tất cả** các từ, nó luôn dài đúng `total` ký tự. Nên ta chỉ cần xét các chuỗi con dài `total` của `s`.

Ô code dưới đây in các chuỗi con dài 6 của Example 1 tại vị trí `0` và `9`, rồi cắt nó thành từng khúc dài 3.

In [1]:
s = "barfoothefoobarman"
words = ["foo", "bar"]

n = len(s)
m = len(words)
k = len(words[0])
total = m * k
print("n =", n, "| m =", m, "| k =", k, "| total =", total)

for start in [0, 9, 3]:
    sub = s[start:start + total]
    pieces = [sub[0:3], sub[3:6]]
    print(f"start={start}: {sub!r} -> {pieces}")

n = 18 | m = 2 | k = 3 | total = 6
start=0: 'barfoo' -> ['bar', 'foo']
start=9: 'foobar' -> ['foo', 'bar']
start=3: 'foothe' -> ['foo', 'the']


Nhận xét:

- Tại `0`: `["bar", "foo"]` — đúng là một hoán vị của `["foo", "bar"]`.
- Tại `9`: `["foo", "bar"]` — cũng đúng.
- Tại `3`: `["foo", "the"]` — có `"the"` lạ, **không** hợp lệ.

Vậy câu hỏi cốt lõi là: **"các khúc cắt ra có đúng là `words` (không quan tâm thứ tự) hay không?"**

Trước khi giải, ta học vài công cụ Python cần dùng.

## 3. Nền tảng Python

### 3.1. Slicing `s[i:j]` — cắt chuỗi con

`s[i:j]` lấy các ký tự từ chỉ số `i` **đến trước** `j` (không lấy `j`). Độ dài kết quả là `j - i`.

- `s[i:i + k]` = đúng `k` ký tự bắt đầu tại `i`.
- Nếu `j` vượt quá độ dài chuỗi, Python **không báo lỗi** mà cắt tới hết — đây là điểm dễ gây bug âm thầm.

In [2]:
s = "barfoothe"
print(s[0:3])     # 'bar'
print(s[3:6])     # 'foo'
print(s[6:9])     # 'the'
print(s[7:10])    # 'he'  <- chỉ còn 2 ký tự, KHÔNG báo lỗi!
print(len(s[2:2 + 3]))  # 3

bar
foo
the
he
3


### 3.2. `range(start, stop, step)` — nhảy cóc từng bước `k`

`range(start, stop, step)` sinh các số `start, start+step, start+2*step, ...` và dừng **trước** `stop`.

Kết hợp với slicing, ta cắt một chuỗi thành các **từ dài `k`** liền nhau:

In [3]:
sub = "foobarthe"
k = 3
print(list(range(0, len(sub), k)))          # [0, 3, 6]

pieces = [sub[j:j + k] for j in range(0, len(sub), k)]
print(pieces)                               # ['foo', 'bar', 'the']

[0, 3, 6]
['foo', 'bar', 'the']


Cú pháp `[biểu_thức for j in ...]` gọi là **list comprehension** (tạo list gọn trong một dòng). Nó tương đương:

```python
pieces = []
for j in range(0, len(sub), k):
    pieces.append(sub[j:j + k])
```

### 3.3. Đếm tần suất bằng `dict`

Vì thứ tự các từ **không quan trọng**, nhưng **số lần** thì quan trọng, công cụ tự nhiên nhất là **bảng đếm** (frequency map): `từ -> số lần xuất hiện`.

Cách tự viết bằng `dict`: `cnt.get(w, 0)` trả về giá trị của key `w`, hoặc `0` nếu chưa có key đó.

In [4]:
words = ["word", "good", "best", "word"]

cnt = {}
for w in words:
    cnt[w] = cnt.get(w, 0) + 1

print(cnt)              # {'word': 2, 'good': 1, 'best': 1}
print(cnt["word"])      # 2
print("bad" in cnt)     # False

{'word': 2, 'good': 1, 'best': 1}
2
False


### 3.4. `collections.Counter` — bảng đếm có sẵn

`Counter` là một loại `dict` chuyên để đếm:

- `Counter(list)` đếm sẵn mọi phần tử.
- Truy cập key chưa có trả về `0` (không báo `KeyError` như `dict`).
- Hai `Counter` so sánh bằng `==` khi **mọi phần tử có cùng số lần** — đúng thứ ta cần để kiểm tra "là hoán vị của nhau".

In [5]:
from collections import Counter

need = Counter(["foo", "bar"])
print(need)                                   # Counter({'foo': 1, 'bar': 1})
print(need["xyz"])                            # 0, không lỗi

print(Counter(["bar", "foo"]) == need)        # True : khác thứ tự vẫn bằng
print(Counter(["foo", "foo"]) == need)        # False: sai số lần
print(set(["foo", "foo", "bar"]) == set(["foo", "bar"]))  # True -> set mất thông tin số lần!

Counter({'foo': 1, 'bar': 1})
0
True
False
True


Dòng cuối rất quan trọng: `set` (tập hợp) **bỏ đi số lần lặp**, nên `{"foo","bar"}` không phân biệt được `["foo","foo","bar"]` với `["foo","bar"]`. Với bài này **phải dùng bảng đếm**, không dùng `set`.

Ta cũng sẽ cộng/trừ từng phần tử trong `Counter` khi trượt cửa sổ:

In [6]:
from collections import Counter

window = Counter()
window["foo"] += 1
window["bar"] += 1
window["foo"] += 1
print(window)          # Counter({'foo': 2, 'bar': 1})

window["foo"] -= 1
print(window)          # Counter({'foo': 1, 'bar': 1})

window.clear()         # xoá sạch
print(window)          # Counter()

Counter({'foo': 2, 'bar': 1})
Counter({'foo': 1, 'bar': 1})
Counter()


## 4. Cách 1 — Brute force: sinh mọi hoán vị (chỉ để hiểu đề)

Đọc đề theo nghĩa đen: "chuỗi ghép = nối các từ theo một hoán vị". Vậy:

1. Sinh **mọi hoán vị** của `words`, nối lại thành chuỗi, cho vào một `set` (để bỏ trùng).
2. Với mỗi vị trí `i`, kiểm tra `s[i:i+total]` có nằm trong `set` đó không.

`itertools.permutations(words)` sinh lần lượt mọi cách sắp xếp:

In [7]:
from itertools import permutations

for p in permutations(["ab", "cd", "ef"]):
    print(p, "->", "".join(p))

('ab', 'cd', 'ef') -> abcdef
('ab', 'ef', 'cd') -> abefcd
('cd', 'ab', 'ef') -> cdabef
('cd', 'ef', 'ab') -> cdefab
('ef', 'ab', 'cd') -> efabcd
('ef', 'cd', 'ab') -> efcdab


Đúng 6 chuỗi như ví dụ trong đề. `"".join(p)` nối các chuỗi trong `p` lại với nhau (không chèn gì ở giữa).

Code cách 1:

In [8]:
from itertools import permutations
from typing import List


def find_substring_permutations(s: str, words: List[str]) -> List[int]:
    k = len(words[0])
    total = len(words) * k
    targets = {"".join(p) for p in permutations(words)}   # set các chuỗi ghép
    result = []
    for i in range(len(s) - total + 1):
        if s[i:i + total] in targets:
            result.append(i)
    return result


print(find_substring_permutations("barfoothefoobarman", ["foo", "bar"]))                       # [0, 9]
print(find_substring_permutations("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]))  # []
print(find_substring_permutations("barfoofoobarthefoobarman", ["bar", "foo", "the"]))          # [6, 9, 12]

[0, 9]
[]
[6, 9, 12]


### Vì sao cách 1 không dùng được?

Số hoán vị của `m` từ là `m!` (`m` giai thừa = `1·2·3···m`). Với `m` tới `5000` theo constraints, con số này **khổng lồ**:

In [9]:
import math

for m in [3, 5, 10, 15, 20]:
    print(f"m = {m:>2}: m! = {math.factorial(m):,}")

# 5000! quá lớn để in ra; ta đếm số chữ số bằng log10: log10(m!) = log10(1) + ... + log10(m)
digits = math.floor(sum(math.log10(i) for i in range(1, 5001))) + 1
print("m = 5000: m! có", f"{digits:,}", "chữ số")

m =  3: m! = 6
m =  5: m! = 120
m = 10: m! = 3,628,800
m = 15: m! = 1,307,674,368,000
m = 20: m! = 2,432,902,008,176,640,000
m = 5000: m! có 16,326 chữ số


- Thời gian: `O(m! · m·k)` để sinh và nối, chưa kể kiểm tra → **bùng nổ tổ hợp**.
- Bộ nhớ: cũng `O(m! · m·k)` cho `set`.

**Chỗ vướng:** ta đang quan tâm **thứ tự** (sinh từng hoán vị), trong khi đề thực ra chỉ hỏi **"có đúng bộ từ đó không"**. Thứ tự không quan trọng ⇒ chỉ cần **đếm**. Đó là cách 2.

## 5. Cách 2 — Mỗi vị trí bắt đầu: cắt thành từ rồi so sánh `Counter`

**Mấu chốt:** "là hoán vị của `words`" ⇔ "bảng đếm của các khúc == bảng đếm của `words`".

Các bước:

1. `need = Counter(words)` — đếm một lần.
2. Với mỗi `i` từ `0` đến `n - total` (**bao gồm** `n - total`, nên dùng `range(n - total + 1)`):
   - lấy `sub = s[i:i+total]`,
   - cắt `sub` thành `m` từ dài `k`,
   - nếu `Counter(các từ) == need` thì ghi nhận `i`.

In [10]:
from collections import Counter
from typing import List


def find_substring_counter(s: str, words: List[str]) -> List[int]:
    k = len(words[0])
    m = len(words)
    total = m * k
    need = Counter(words)
    result = []
    for i in range(len(s) - total + 1):
        pieces = [s[i + j * k: i + (j + 1) * k] for j in range(m)]
        if Counter(pieces) == need:
            result.append(i)
    return result


print(find_substring_counter("barfoothefoobarman", ["foo", "bar"]))                       # [0, 9]
print(find_substring_counter("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]))  # []
print(find_substring_counter("barfoofoobarthefoobarman", ["bar", "foo", "the"]))          # [6, 9, 12]

[0, 9]
[]
[6, 9, 12]


### Trace cách 2 trên Example 1

Hàm `trace_counter` in từng vị trí `i`, các từ cắt được và kết luận. Để ý: với `n = 18`, `total = 6` có `18 - 6 + 1 = 13` vị trí cần thử.

In [11]:
def trace_counter(s, words):
    k, m = len(words[0]), len(words)
    total = m * k
    need = Counter(words)
    print("need =", dict(need))
    for i in range(len(s) - total + 1):
        pieces = [s[i + j * k: i + (j + 1) * k] for j in range(m)]
        ok = Counter(pieces) == need
        print(f"i={i:>2} pieces={pieces} -> {'KHỚP' if ok else '-'}")


trace_counter("barfoothefoobarman", ["foo", "bar"])

need = {'foo': 1, 'bar': 1}
i= 0 pieces=['bar', 'foo'] -> KHỚP
i= 1 pieces=['arf', 'oot'] -> -
i= 2 pieces=['rfo', 'oth'] -> -
i= 3 pieces=['foo', 'the'] -> -
i= 4 pieces=['oot', 'hef'] -> -
i= 5 pieces=['oth', 'efo'] -> -
i= 6 pieces=['the', 'foo'] -> -
i= 7 pieces=['hef', 'oob'] -> -
i= 8 pieces=['efo', 'oba'] -> -
i= 9 pieces=['foo', 'bar'] -> KHỚP
i=10 pieces=['oob', 'arm'] -> -
i=11 pieces=['oba', 'rma'] -> -
i=12 pieces=['bar', 'man'] -> -


### Độ phức tạp cách 2

- Có `n - total + 1` vị trí (≈ `n`), mỗi vị trí cắt `m` từ, mỗi từ tốn `O(k)` để cắt + băm (hash).
- **Time:** `O((n - m·k + 1) · m · k)`. Với `n = 10^4`, `m·k` có thể vài nghìn ⇒ cỡ hàng chục triệu thao tác ký tự — chậm.
- **Space:** `O(m · k)` cho `need` và bảng đếm tạm.

### Cách 2 vướng ở đâu?

Nhìn lại trace: vị trí `i = 0` cắt ra `bar | foo`, vị trí `i = 3` cắt ra `foo | the`. Từ `"foo"` (tại chỉ số 3) bị cắt và đếm **lại từ đầu** ở cả hai lần.

Hai cửa sổ `i` và `i + k` **chồng lên nhau** `m - 1` từ, nhưng cách 2 vứt hết và đếm lại. Nếu ta **tái sử dụng** bảng đếm khi dịch cửa sổ sang phải đúng **một từ** (`k` ký tự), mỗi bước chỉ cần **thêm 1 từ bên phải, bớt từ bên trái**. Đó là cách 3.

## 6. Bước đột phá — Chia `s` thành `k` "làn" theo offset

Câu hỏi: nếu cửa sổ dịch đúng `k` ký tự mỗi lần, thì các vị trí `i = 1, 2, ...` (không chia hết cho `k`) xử lý thế nào?

Quan sát: một chuỗi ghép bắt đầu tại `i` thì các từ của nó nằm ở `i, i+k, i+2k, ...` — **tất cả có cùng số dư khi chia cho `k`**. Số dư đó gọi là **offset** (`offset = i % k`, chỉ có `k` giá trị `0, 1, ..., k-1`).

Vậy ta tách `s` thành `k` "làn" (lane). Làn `offset` là dãy từ cắt tại `offset, offset+k, offset+2k, ...`. Mỗi chuỗi ghép **nằm gọn trong đúng một làn**, ở dạng `m` từ **liên tiếp** trong làn đó.

`%` là phép chia lấy dư: `9 % 3 = 0`, `7 % 3 = 1`.

In [12]:
s = "barfoothefoobarman"
k = 3

for offset in range(k):
    lane = [(i, s[i:i + k]) for i in range(offset, len(s) - k + 1, k)]
    print(f"offset {offset}:", lane)

offset 0: [(0, 'bar'), (3, 'foo'), (6, 'the'), (9, 'foo'), (12, 'bar'), (15, 'man')]
offset 1: [(1, 'arf'), (4, 'oot'), (7, 'hef'), (10, 'oob'), (13, 'arm')]
offset 2: [(2, 'rfo'), (5, 'oth'), (8, 'efo'), (11, 'oba'), (14, 'rma')]


Ở làn `offset 0` ta thấy dãy từ: `bar foo the foo bar man`. Đáp án `0` là cặp `(bar, foo)` liên tiếp; đáp án `9` là cặp `(foo, bar)` liên tiếp. Hai làn còn lại toàn "từ rác".

Lưu ý điều kiện dừng `len(s) - k + 1`: chỉ cắt khi còn **đủ `k` ký tự** (nhớ mục 3.1 — slicing không báo lỗi khi thiếu ký tự, nên ta phải tự chặn).

**Bài toán đã đổi thành:** trên mỗi làn (một list các từ), tìm mọi **đoạn liên tiếp dài `m` từ** có bảng đếm đúng bằng `need`. Đây là bài kinh điển của **sliding window**.

## 7. Nền tảng — Sliding window (cửa sổ trượt) là gì?

**Sliding window** là kỹ thuật dùng hai chỉ số `left` và `right` đánh dấu một **đoạn liên tiếp** `[left, right]` của dãy:

- `right` luôn tiến lên 1 bước: **mở rộng** cửa sổ, thêm phần tử mới vào trạng thái (bảng đếm).
- Nếu cửa sổ **vi phạm** điều kiện, tăng `left` để **co** cửa sổ, bớt phần tử bên trái khỏi trạng thái, cho tới khi hợp lệ lại.
- Vì `left` và `right` **chỉ tiến, không lùi**, tổng số bước của cả hai là `O(độ dài dãy)`.

### Bài khởi động

Cho một list token, tìm đoạn liên tiếp **dài nhất mà không token nào lặp lại**. Điều kiện vi phạm: có token xuất hiện `> 1` lần trong cửa sổ.

In [13]:
from collections import Counter


def longest_no_repeat(tokens):
    window = Counter()
    left = 0
    best = 0
    for right, t in enumerate(tokens):
        window[t] += 1                      # mở rộng: thêm token bên phải
        while window[t] > 1:                # vi phạm: t bị lặp
            window[tokens[left]] -= 1       # co: bỏ token bên trái
            left += 1
        best = max(best, right - left + 1)
        print(f"right={right} t={t!r:6} left={left} cửa sổ={tokens[left:right + 1]}")
    return best


print("dài nhất =", longest_no_repeat(["a", "b", "a", "c", "b"]))

right=0 t='a'    left=0 cửa sổ=['a']
right=1 t='b'    left=0 cửa sổ=['a', 'b']
right=2 t='a'    left=1 cửa sổ=['b', 'a']
right=3 t='c'    left=1 cửa sổ=['b', 'a', 'c']
right=4 t='b'    left=2 cửa sổ=['a', 'c', 'b']
dài nhất = 3


`enumerate(tokens)` cho từng cặp `(chỉ số, phần tử)`.

Quan sát bước `right=2`: thêm `'a'` làm `'a'` xuất hiện 2 lần → co từ trái: bỏ `'a'` ở `left=0` → `left=1`, hết vi phạm.

**Liên hệ với bài 30:** trên mỗi làn, ta làm y hệt, chỉ đổi:

| Bài khởi động | Bài 30 |
|---|---|
| token = 1 phần tử list | token = 1 từ dài `k`, cắt bằng `s[right:right+k]` |
| vi phạm khi `window[t] > 1` | vi phạm khi `window[w] > need[w]` (thừa so với yêu cầu) |
| — | gặp từ **không có trong `need`** ⇒ không cửa sổ nào chứa nó được ⇒ **xoá sạch**, nhảy `left` qua nó |
| ghi nhận độ dài lớn nhất | ghi nhận `left` khi cửa sổ có đủ `m` từ |
| `left`, `right` tăng 1 | `left`, `right` tăng `k` (đơn vị là ký tự trong `s`) |

## 8. Cách 3 (cách chính) — Sliding window theo từng offset

### Trạng thái cho mỗi làn

- `left`: chỉ số (trong `s`) của từ đầu tiên trong cửa sổ.
- `window`: `Counter` đếm các từ trong cửa sổ.
- `count`: số từ trong cửa sổ (= tổng các giá trị trong `window`).

### Với mỗi `right = offset, offset+k, ...` (từ mới `word = s[right:right+k]`)

1. **Từ lạ** (`word not in need`): mọi cửa sổ chứa nó đều hỏng ⇒ `window.clear()`, `count = 0`, `left = right + k` (bắt đầu lại ngay sau từ lạ).
2. **Từ hợp lệ**: thêm vào: `window[word] += 1`, `count += 1`.
3. **Co nếu thừa**: `while window[word] > need[word]`: bỏ từ tại `left` khỏi cửa sổ, `left += k`, `count -= 1`.
4. **Ghi nhận**: nếu `count == m` thì cửa sổ chính là một chuỗi ghép ⇒ thêm `left` vào kết quả.

Chỉ cần kiểm tra `word` ở bước 3, vì chỉ có `word` vừa tăng nên chỉ nó có thể vừa bị thừa.

In [14]:
from collections import Counter
from typing import List


def find_substring_sliding(s: str, words: List[str]) -> List[int]:
    n, m, k = len(s), len(words), len(words[0])
    if n < m * k:
        return []
    need = Counter(words)
    result = []

    for offset in range(k):                       # k làn
        left = offset
        window = Counter()
        count = 0
        for right in range(offset, n - k + 1, k):  # right nhảy từng từ
            word = s[right:right + k]

            if word not in need:                   # (1) từ lạ -> làm lại từ đầu
                window.clear()
                count = 0
                left = right + k
                continue

            window[word] += 1                      # (2) thêm từ mới
            count += 1

            while window[word] > need[word]:       # (3) thừa -> co từ trái
                left_word = s[left:left + k]
                window[left_word] -= 1
                count -= 1
                left += k

            if count == m:                         # (4) đủ m từ -> khớp
                result.append(left)

    return result


print(find_substring_sliding("barfoothefoobarman", ["foo", "bar"]))                       # [0, 9]
print(find_substring_sliding("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]))  # []
print(find_substring_sliding("barfoofoobarthefoobarman", ["bar", "foo", "the"]))          # [6, 9, 12]

[0, 9]
[]
[6, 9, 12]


### Câu hỏi hay: sau khi khớp, có cần chủ động dịch `left` không?

Không cần. Khi `count == m`, cửa sổ **đúng bằng** `need`. Từ hợp lệ kế tiếp `w` sẽ làm `window[w] = need[w] + 1` ⇒ vòng `while` ở bước 3 tự co cửa sổ. Từ lạ kế tiếp thì bước 1 xoá sạch. Vì vậy code không cần nhánh đặc biệt sau khi khớp — trace dưới đây sẽ cho thấy điều đó.



## 9. Trace cách 3 trên Example 3

`s = "barfoofoobarthefoobarman"`, `words = ["bar","foo","the"]`, `k = 3`, `m = 3`, `need = {bar:1, foo:1, the:1}`.

Làn `offset 0` gồm các từ: `bar(0) foo(3) foo(6) bar(9) the(12) foo(15) bar(18) man(21)`. Input này đủ "khó" vì có cả 3 nhánh: **co cửa sổ** (hai `foo` liền nhau), **khớp liên tiếp** và **từ lạ**.

| right | word | Hành động | left sau | window sau | count | Ghi nhận |
|---:|---|---|---:|---|---:|---|
| 0 | bar | thêm | 0 | {bar:1} | 1 | |
| 3 | foo | thêm | 0 | {bar:1, foo:1} | 2 | |
| 6 | foo | thêm → foo thừa (2 > 1): bỏ `bar`@0, bỏ `foo`@3 | 6 | {foo:1} | 1 | |
| 9 | bar | thêm | 6 | {bar:1, foo:1} | 2 | |
| 12 | the | thêm | 6 | {bar:1, foo:1, the:1} | 3 | **6** |
| 15 | foo | thêm → foo thừa: bỏ `foo`@6 | 9 | {bar:1, foo:1, the:1} | 3 | **9** |
| 18 | bar | thêm → bar thừa: bỏ `bar`@9 | 12 | {bar:1, foo:1, the:1} | 3 | **12** |
| 21 | man | từ lạ → xoá sạch | 24 | {} | 0 | |

Ý nghĩa cột: `right` là chỉ số từ mới trong `s`; "left sau / window sau / count" là trạng thái **sau** khi xử lý xong bước đó. Ở 3 dòng khớp, cửa sổ luôn có đủ `bar, foo, the`, chỉ là vị trí thay đổi: `foobarthe` → `barthefoo` → `thefoobar`.

Hàm `trace_sliding` dưới đây in đúng các bước này (chỉ in làn được chọn để output gọn).

In [15]:
def fmt(counter):
    return "{" + ", ".join(f"{w}:{c}" for w, c in counter.items() if c > 0) + "}"


def trace_sliding(s, words, offsets=None):
    n, m, k = len(s), len(words), len(words[0])
    need = Counter(words)
    print("need =", fmt(need), "| m =", m, "| k =", k)
    result = []
    for offset in (range(k) if offsets is None else offsets):
        print(f"--- offset {offset} ---")
        left, window, count = offset, Counter(), 0
        for right in range(offset, n - k + 1, k):
            word = s[right:right + k]
            if word not in need:
                window.clear(); count = 0; left = right + k
                print(f"right={right:>2} {word!r}: từ lạ -> xoá sạch, left={left}")
                continue
            window[word] += 1; count += 1
            removed = []
            while window[word] > need[word]:
                lw = s[left:left + k]
                window[lw] -= 1; count -= 1
                removed.append(f"{lw}@{left}")
                left += k
            note = f" | thừa {word!r}, bỏ {removed}" if removed else ""
            hit = f"  ==> KHỚP tại {left}" if count == m else ""
            print(f"right={right:>2} {word!r}: left={left:>2} window={fmt(window)} count={count}{note}{hit}")
            if count == m:
                result.append(left)
    print("kết quả:", result)


trace_sliding("barfoofoobarthefoobarman", ["bar", "foo", "the"], offsets=[0])

need = {bar:1, foo:1, the:1} | m = 3 | k = 3
--- offset 0 ---
right= 0 'bar': left= 0 window={bar:1} count=1
right= 3 'foo': left= 0 window={bar:1, foo:1} count=2
right= 6 'foo': left= 6 window={foo:1} count=1 | thừa 'foo', bỏ ['bar@0', 'foo@3']
right= 9 'bar': left= 6 window={bar:1, foo:1} count=2
right=12 'the': left= 6 window={bar:1, foo:1, the:1} count=3  ==> KHỚP tại 6
right=15 'foo': left= 9 window={bar:1, foo:1, the:1} count=3 | thừa 'foo', bỏ ['foo@6']  ==> KHỚP tại 9
right=18 'bar': left=12 window={bar:1, foo:1, the:1} count=3 | thừa 'bar', bỏ ['bar@9']  ==> KHỚP tại 12
right=21 'man': từ lạ -> xoá sạch, left=24
kết quả: [6, 9, 12]


Output khớp bảng trên. Hai làn còn lại (`offset 1` và `offset 2`) toàn từ lạ như `"arf"`, `"oof"`, ... nên mỗi bước đều xoá sạch — ta in nhanh để thấy:

In [16]:
trace_sliding("barfoofoobarthefoobarman", ["bar", "foo", "the"], offsets=[1, 2])

need = {bar:1, foo:1, the:1} | m = 3 | k = 3
--- offset 1 ---
right= 1 'arf': từ lạ -> xoá sạch, left=4
right= 4 'oof': từ lạ -> xoá sạch, left=7
right= 7 'oob': từ lạ -> xoá sạch, left=10
right=10 'art': từ lạ -> xoá sạch, left=13
right=13 'hef': từ lạ -> xoá sạch, left=16
right=16 'oob': từ lạ -> xoá sạch, left=19
right=19 'arm': từ lạ -> xoá sạch, left=22
--- offset 2 ---
right= 2 'rfo': từ lạ -> xoá sạch, left=5
right= 5 'ofo': từ lạ -> xoá sạch, left=8
right= 8 'oba': từ lạ -> xoá sạch, left=11
right=11 'rth': từ lạ -> xoá sạch, left=14
right=14 'efo': từ lạ -> xoá sạch, left=17
right=17 'oba': từ lạ -> xoá sạch, left=20
right=20 'rma': từ lạ -> xoá sạch, left=23
kết quả: []


### Trace với từ lặp trong `words` (Example 2)

`words = ["word","good","best","word"]` ⇒ `need = {word:2, good:1, best:1}`. Từ `"good"` chỉ được phép 1 lần, nên mỗi lần gặp `good` thứ hai, cửa sổ bị co mạnh. Kết quả rỗng như đề.

In [17]:
trace_sliding("wordgoodgoodgoodbestword", ["word", "good", "best", "word"], offsets=[0])

need = {word:2, good:1, best:1} | m = 4 | k = 4
--- offset 0 ---
right= 0 'word': left= 0 window={word:1} count=1
right= 4 'good': left= 0 window={word:1, good:1} count=2
right= 8 'good': left= 8 window={good:1} count=1 | thừa 'good', bỏ ['word@0', 'good@4']
right=12 'good': left=12 window={good:1} count=1 | thừa 'good', bỏ ['good@8']
right=16 'best': left=12 window={good:1, best:1} count=2
right=20 'word': left=12 window={word:1, good:1, best:1} count=3
kết quả: []


Đi theo từng dòng:

- `right=8`: `good` thứ hai ⇒ thừa ⇒ bỏ `word`@0 rồi `good`@4 ⇒ cửa sổ chỉ còn `{good:1}`, `left=8`.
- `right=12`: lại thừa `good` ⇒ bỏ `good`@8 ⇒ `left=12`.
- Cuối cùng cửa sổ `good best word` chỉ có 3 từ, thiếu một `word` ⇒ không bao giờ đạt `count == 4`.

(Các làn `offset 1, 2, 3` toàn từ lạ, không cần in.)

## 10. Code nộp LeetCode — Cách 3 (sliding window)

Trên LeetCode, `List` và `Counter` thường đã được import sẵn; ta vẫn ghi import để notebook chạy độc lập.

In [18]:
from collections import Counter
from typing import List


class Solution:
    def findSubstring(self, s: str, words: List[str]) -> List[int]:
        n, m, k = len(s), len(words), len(words[0])
        if n < m * k:
            return []
        need = Counter(words)
        result = []

        for offset in range(k):
            left = offset
            window = Counter()
            count = 0
            for right in range(offset, n - k + 1, k):
                word = s[right:right + k]
                if word not in need:
                    window.clear()
                    count = 0
                    left = right + k
                    continue
                window[word] += 1
                count += 1
                while window[word] > need[word]:
                    window[s[left:left + k]] -= 1
                    count -= 1
                    left += k
                if count == m:
                    result.append(left)

        return result

### Bộ test

Đề cho phép trả về theo **thứ tự bất kỳ**, nên ta so sánh sau khi `sorted(...)`.

Các nhóm test:

1. 3 ví dụ của đề.
2. `s` ngắn hơn `total` ⇒ `[]`.
3. Một từ duy nhất (`m = 1`) ⇒ mọi vị trí xuất hiện của từ đó.
4. Từ lặp, chuỗi toàn một ký tự (nhiều đáp án chồng lên nhau).
5. Đáp án nằm ở làn `offset != 0`.
6. Từ lạ chen giữa (kiểm tra nhánh xoá sạch).
7. `words` có từ lặp nhưng `s` thiếu một bản.

In [19]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"{name:<32} PASS  {sorted(actual)}")


sol = Solution()
check("example 1", sol.findSubstring("barfoothefoobarman", ["foo", "bar"]), [0, 9])
check("example 2", sol.findSubstring("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]), [])
check("example 3", sol.findSubstring("barfoofoobarthefoobarman", ["bar", "foo", "the"]), [6, 9, 12])
check("s ngắn hơn total", sol.findSubstring("ab", ["ab", "ab"]), [])
check("m = 1", sol.findSubstring("abcabc", ["bc"]), [1, 4])
check("toàn 'a', từ lặp", sol.findSubstring("aaaaaa", ["aa", "aa"]), [0, 1, 2])
check("đáp án ở offset 1", sol.findSubstring("xbarfoo", ["bar", "foo"]), [1])
check("từ lạ chen giữa", sol.findSubstring("barxxxfoo", ["bar", "foo"]), [])
check("words lặp, s thiếu", sol.findSubstring("foobarbar", ["foo", "bar", "foo"]), [])
check("k = 1", sol.findSubstring("abba", ["a", "b"]), [0, 2])
check("đúng bằng cả chuỗi", sol.findSubstring("foobar", ["bar", "foo"]), [0])
print("Tất cả test đều PASS!")

example 1                        PASS  [0, 9]
example 2                        PASS  []
example 3                        PASS  [6, 9, 12]
s ngắn hơn total                 PASS  []
m = 1                            PASS  [1, 4]
toàn 'a', từ lặp                 PASS  [0, 1, 2]
đáp án ở offset 1                PASS  [1]
từ lạ chen giữa                  PASS  []
words lặp, s thiếu               PASS  []
k = 1                            PASS  [0, 2]
đúng bằng cả chuỗi               PASS  [0]
Tất cả test đều PASS!


### Test đối chiếu chéo 3 cách trên input ngẫu nhiên

Sinh `s` và `words` từ bảng chữ cái nhỏ `"ab"` để các từ hay trùng nhau (dễ có đáp án, dễ lộ bug). Giữ `m <= 4` để cách hoán vị (`m!`) vẫn chạy được. `random.seed(30)` cố định để mỗi lần chạy đều ra cùng dữ liệu.

In [20]:
import random

random.seed(30)
found = 0
for _ in range(500):
    k = random.randint(1, 3)
    m = random.randint(1, 4)
    words = ["".join(random.choice("ab") for _ in range(k)) for _ in range(m)]
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 16)))

    a = sorted(find_substring_permutations(s, words))
    b = sorted(find_substring_counter(s, words))
    c = sorted(find_substring_sliding(s, words))
    d = sorted(Solution().findSubstring(s, words))
    assert a == b == c == d, (s, words, a, b, c, d)
    found += bool(a)

print("500 bộ test ngẫu nhiên: 4 hàm cho kết quả giống nhau.")
print("Số bộ có ít nhất một đáp án:", found)

500 bộ test ngẫu nhiên: 4 hàm cho kết quả giống nhau.
Số bộ có ít nhất một đáp án: 208


## 11. Vì sao cách 3 đúng?

Xét một làn `offset` cố định. Sau mỗi bước xử lý `right`, các **bất biến** (invariant — điều luôn đúng) sau được giữ:

1. **Liên tiếp:** cửa sổ là các từ tại `left, left+k, ..., right` (hoặc rỗng nếu vừa gặp từ lạ), **không chứa từ lạ**.
2. **Không thừa:** với mọi từ `w`, `window[w] <= need[w]`.
3. **`count`** bằng số từ trong cửa sổ.

Từ đó:

- **Ghi nhận đúng (không trả nhầm):** nếu `count == m` thì tổng `window` = `m` = tổng `need`, mà từng `window[w] <= need[w]` ⇒ buộc phải `window[w] == need[w]` với mọi `w` ⇒ cửa sổ dài đúng `m` từ và là một hoán vị của `words`.
- **Không bỏ sót:** giả sử tại `p` có một chuỗi ghép; nó nằm ở làn `p % k`. Xét lúc `right` tới từ cuối của nó, `right = p + (m-1)k`.
  - `left <= p`: `left` chỉ vượt qua `p` nếu (a) gặp từ lạ trong đoạn `p..right` — không có, vì đoạn này toàn từ của `words`; hoặc (b) vòng `while` co bỏ luôn từ tại `p` — nhưng khi `left == p` thì cửa sổ là một phần của chuỗi ghép, không thể thừa, nên `while` dừng trước đó.
  - `left >= p`: theo bất biến 2, cửa sổ có tối đa `m` từ, nên `left >= right - (m-1)k = p`.
  - Vậy `left == p`, cửa sổ đúng bằng chuỗi ghép, `count == m` ⇒ `p` được ghi nhận.
- **Không ghi trùng:** mỗi bước `right` ghi tối đa một `left`, và `left` chỉ tăng ⇒ mỗi `p` xuất hiện tối đa một lần.

Ghép `k` làn lại thì phủ mọi chỉ số bắt đầu `0..n-1`.

## 12. Độ phức tạp

Ký hiệu: `n = len(s)`, `m = len(words)`, `k = len(words[0])`.

**Cách 3:**

- Với mỗi trong `k` làn, `right` đi khoảng `n / k` bước; `left` cũng chỉ tiến và đi tối đa `n / k` bước (đó là lý do vòng `while` lồng bên trong **không** làm thành bình phương).
- Mỗi bước cắt một từ `s[x:x+k]` và băm nó: `O(k)`.
- Tổng: `k · (n/k + n/k) · O(k)` = **`O(n · k)`**, cộng `O(m · k)` để dựng `need`.
- **Time:** `O(n·k + m·k)`. **Space:** `O(m·k)` cho `need` và `window` (mỗi bảng tối đa `m` key dài `k`).

Với constraints: `n·k <= 10^4 · 30 = 3·10^5` — rất nhanh.

### So sánh 3 cách

| Cách | Ý tưởng | Time | Space | Dùng khi nào |
|---|---|---|---|---|
| 1. Hoán vị | Sinh mọi chuỗi ghép rồi tra `set` | `O(m! · m·k)` | `O(m! · m·k)` | Chỉ để hiểu đề |
| 2. Counter mỗi vị trí | Cắt `m` từ ở mỗi `i`, so `Counter` | `O(n · m · k)` | `O(m·k)` | Lời giải đầu tiên, dễ viết, dùng làm brute force kiểm tra |
| 3. Sliding window theo offset | `k` làn, thêm phải / co trái | `O(n · k + m·k)` | `O(m·k)` | Lời giải chính, nên nộp |

Đo thử thời gian (con số cụ thể tuỳ máy, chỉ cần nhìn **tỉ lệ**):

In [21]:
import time

random.seed(1)
s_big = "".join(random.choice("ab") for _ in range(10_000))
words_big = ["".join(random.choice("ab") for _ in range(3)) for _ in range(200)]

t0 = time.perf_counter(); r2 = find_substring_counter(s_big, words_big); t1 = time.perf_counter()
r3 = Solution().findSubstring(s_big, words_big); t2 = time.perf_counter()

assert sorted(r2) == sorted(r3)
print(f"n = {len(s_big)}, m = {len(words_big)}, k = 3, số đáp án = {len(r3)}")
print(f"Cách 2 (Counter mỗi vị trí): {t1 - t0:.3f} s")
print(f"Cách 3 (sliding window)    : {t2 - t1:.3f} s")

n = 10000, m = 200, k = 3, số đáp án = 0
Cách 2 (Counter mỗi vị trí): 0.293 s
Cách 3 (sliding window)    : 0.008 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ trượt ở offset 0

Sai:

```python
for right in range(0, n - k + 1, k):   # quên vòng for offset in range(k)
```

Vì sao sai: chuỗi ghép có thể bắt đầu ở vị trí không chia hết cho `k` (ví dụ `"xbarfoo"`, đáp án `1`). Sửa: lặp `for offset in range(k)`.

### Lỗi 2: Gặp từ lạ mà chỉ `continue`, không xoá cửa sổ

Sai:

```python
if word not in need:
    continue            # cửa sổ vẫn giữ các từ cũ!
```

Vì sao sai: cửa sổ không còn **liên tiếp** — nó "nhảy qua" từ lạ. Với `"barxxxfoo"`, `["bar","foo"]` sẽ ghi nhầm vị trí `0`.

Ô code dưới chạy thử hai phiên bản sai này:

In [22]:
def sliding_wrong(s, words, all_offsets=True, reset_on_unknown=True):
    n, m, k = len(s), len(words), len(words[0])
    need = Counter(words)
    result = []
    for offset in (range(k) if all_offsets else [0]):
        left, window, count = offset, Counter(), 0
        for right in range(offset, n - k + 1, k):
            word = s[right:right + k]
            if word not in need:
                if reset_on_unknown:
                    window.clear(); count = 0; left = right + k
                continue
            window[word] += 1; count += 1
            while window[word] > need[word]:
                window[s[left:left + k]] -= 1; count -= 1; left += k
            if count == m:
                result.append(left)
    return result


print("Lỗi 1:", sliding_wrong("xbarfoo", ["bar", "foo"], all_offsets=False),
      "| đúng:", find_substring_sliding("xbarfoo", ["bar", "foo"]))
print("Lỗi 2:", sliding_wrong("barxxxfoo", ["bar", "foo"], reset_on_unknown=False),
      "| đúng:", find_substring_sliding("barxxxfoo", ["bar", "foo"]))

Lỗi 1: [] | đúng: [1]
Lỗi 2: [0] | đúng: []


### Lỗi 3: Dùng `set` thay cho bảng đếm

Sai:

```python
if set(pieces) == set(words):
```

Vì sao sai: `set` mất thông tin số lần. Với `words = ["foo","bar","foo"]` và `s = "foobarbar"`, `set(["foo","bar","bar"]) == {"foo","bar"}` ⇒ ghi nhầm `0`. Sửa: `Counter(pieces) == Counter(words)`.

### Lỗi 4: Sai biên `range` ở cách 2

Sai:

```python
for i in range(len(s) - total):        # thiếu + 1
```

Vì sao sai: bỏ mất vị trí cuối cùng `i = n - total`. Với `s = "foobar"`, `words = ["bar","foo"]` thì `range(0)` rỗng ⇒ trả `[]` thay vì `[0]`. Sửa: `range(len(s) - total + 1)`.

In [23]:
def counter_wrong_set(s, words):
    k, m = len(words[0]), len(words)
    total = m * k
    return [i for i in range(len(s) - total + 1)
            if set(s[i + j * k: i + (j + 1) * k] for j in range(m)) == set(words)]


def counter_wrong_range(s, words):
    k, m = len(words[0]), len(words)
    total = m * k
    need = Counter(words)
    return [i for i in range(len(s) - total)
            if Counter(s[i + j * k: i + (j + 1) * k] for j in range(m)) == need]


print("Lỗi 3:", counter_wrong_set("foobarbar", ["foo", "bar", "foo"]),
      "| đúng:", find_substring_counter("foobarbar", ["foo", "bar", "foo"]))
print("Lỗi 4:", counter_wrong_range("foobar", ["bar", "foo"]),
      "| đúng:", find_substring_counter("foobar", ["bar", "foo"]))

Lỗi 3: [0] | đúng: []
Lỗi 4: [] | đúng: [0]


### Lỗi 5: Quên `count -= 1` (hoặc quên `window[...] -= 1`) khi co cửa sổ

Hai biến `window` và `count` phải **luôn đi cùng nhau**: mỗi lần `left += k` thì phải bớt cả hai. Nếu quên một trong hai, bất biến 2 hoặc 3 ở mục 11 bị phá, và `count == m` sẽ đúng/sai lung tung.

Mẹo tránh: viết 3 dòng co cửa sổ thành một khối cố định, luôn đủ bộ:

```python
window[s[left:left + k]] -= 1
count -= 1
left += k
```

### Lỗi 6: Cho `right` chạy tới `n` thay vì `n - k + 1`

`s[right:right+k]` ở gần cuối chuỗi sẽ **ngắn hơn `k`** mà không báo lỗi (mục 3.1). Thường nó không nằm trong `need` nên "vô tình" vẫn đúng, nhưng đây là thói quen nguy hiểm. Luôn dừng ở `n - k + 1`.

## 14. Mẹo nhận dạng pattern

Nghĩ tới **sliding window + bảng đếm** khi đề có các dấu hiệu:

- "Tìm **đoạn con liên tiếp** (substring / subarray) ..."
- "... là **hoán vị** / **anagram** của ..." ⇒ thứ tự không quan trọng, chỉ **số lần** quan trọng ⇒ `Counter`.
- Độ dài cửa sổ cố định hoặc có điều kiện "không được thừa".

Riêng bài 30 có thêm dấu hiệu "**mọi từ cùng độ dài `k`**" ⇒ chia `s` thành `k` làn theo `i % k`, rồi coi mỗi từ như một "ký tự lớn".

## 15. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| 438. Find All Anagrams in a String | Chính là bài 30 với `k = 1` (mỗi "từ" là một ký tự) |
| 567. Permutation in String | Như 438 nhưng chỉ cần trả `True/False` |
| 3. Longest Substring Without Repeating Characters | Bài khởi động ở mục 7 (co khi thừa) |
| 76. Minimum Window Substring | Cửa sổ trượt + bảng đếm, tìm cửa sổ ngắn nhất **chứa** đủ |
| 30 với cửa sổ cố định | Thay vì co bằng `while`, luôn giữ đúng `m` từ và đếm số từ "đã đủ" (bài tập 6) |

## 16. Bài tập vận dụng

### Bài 1 — Đoán output

Không chạy code, hãy đoán kết quả `find_substring_sliding` cho các input sau:

```python
find_substring_sliding("foobarfoobar", ["foo", "bar"])
find_substring_sliding("abababab", ["ab", "ba"])
find_substring_sliding("aaa", ["a", "a"])
```

Gợi ý mức 1: tính `total` trước, liệt kê mọi chuỗi con dài `total`.
Gợi ý mức 2: với câu 2, `"ab"` và `"ba"` mỗi từ chỉ 1 lần; thử cả các vị trí lẻ.
Gợi ý mức 3: câu 3 có `total = 2`, các vị trí `0` và `1` đều cho `"aa"`.



### Bài 2 — Tính tay bảng trace

Cho `s = "aaabaaab"`, `words = ["aa", "ab"]` (`k = 2`, `m = 2`).

1. Liệt kê các từ của làn `offset 0` và làn `offset 1`.
2. Lập bảng giống mục 9 (`right`, `word`, hành động, `left`, `window`, `count`, ghi nhận) cho **cả hai làn**.
3. Kết quả cuối cùng là gì?

Gợi ý mức 1: làn 0 cắt tại `0, 2, 4, 6`; làn 1 cắt tại `1, 3, 5`.
Gợi ý mức 2: ở làn 0, từ `"aa"` thứ hai liền nhau sẽ gây **thừa**.
Gợi ý mức 3: làn 1 có từ `"ba"` không nằm trong `need` ⇒ xoá sạch.

### Bài 3 — Điền TODO: cách 2 có "dừng sớm"

Cải tiến cách 2: thay vì cắt đủ `m` từ rồi mới so `Counter`, hãy đếm dần và **dừng ngay** khi gặp từ lạ hoặc từ bị thừa.

In [24]:
def find_substring_counter_early(s, words):
    k, m = len(words[0]), len(words)
    total = m * k
    need = Counter(words)
    result = []
    for i in range(len(s) - total + 1):
        seen = Counter()
        ok = True
        for j in range(m):
            # TODO 1: cắt từ thứ j bắt đầu tại i + j * k
            # TODO 2: nếu từ không có trong need -> ok = False, break
            # TODO 3: tăng seen[word]; nếu seen[word] > need[word] -> ok = False, break
            pass
        # TODO 4: nếu ok thì thêm i vào result
    return result

# Tự làm xong rồi mới mở test:
# assert find_substring_counter_early("barfoothefoobarman", ["foo", "bar"]) == [0, 9]
# assert find_substring_counter_early("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []
# assert find_substring_counter_early("barfoofoobarthefoobarman", ["bar", "foo", "the"]) == [6, 9, 12]

Gợi ý mức 1: logic giống hệt bước (1) và (3) của cách 3, chỉ là không tái sử dụng giữa các `i`.
Gợi ý mức 2: `word = s[i + j * k : i + (j + 1) * k]`.
Gợi ý mức 3: vì đã chặn "không thừa" và đếm đủ `m` từ, nếu vòng `for j` chạy hết mà `ok` vẫn `True` thì chắc chắn khớp.

### Bài 4 — Tự viết lại cách 3

Không nhìn mục 8, tự viết lại hàm sliding window. Chỉ được nhìn khung dưới.

In [25]:
def my_find_substring(s, words):
    # TODO 1: tính n, m, k; tạo need = Counter(words)
    # TODO 2: for offset in range(k): khởi tạo left, window, count
    # TODO 3: for right in range(offset, n - k + 1, k): cắt word
    # TODO 4: từ lạ -> xoá sạch, left = right + k, continue
    # TODO 5: thêm word; while thừa -> co từ trái
    # TODO 6: count == m -> ghi nhận left
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert sorted(my_find_substring("barfoofoobarthefoobarman", ["bar", "foo", "the"])) == [6, 9, 12]
# assert sorted(my_find_substring("aaaaaa", ["aa", "aa"])) == [0, 1, 2]
# assert my_find_substring("xbarfoo", ["bar", "foo"]) == [1]

Gợi ý mức 1: có đúng 4 nhánh: từ lạ / thêm / co / ghi nhận.
Gợi ý mức 2: `left` bắt đầu bằng `offset`, không phải `0`.
Gợi ý mức 3: co cửa sổ luôn là bộ 3 dòng `window[s[left:left+k]] -= 1; count -= 1; left += k`.

### Bài 5 — Biến thể: LeetCode 438 Find All Anagrams in a String

Cho `s` và `p`, trả về mọi chỉ số bắt đầu của các chuỗi con của `s` là **anagram** (hoán vị ký tự) của `p`.

Ví dụ: `s = "cbaebabacd"`, `p = "abc"` ⇒ `[0, 6]`.

Đây là bài 30 với mỗi "từ" là **một ký tự** (`k = 1`), nên chỉ có một làn.

In [26]:
def find_anagrams(s: str, p: str) -> list[int]:
    # TODO: sliding window với need = Counter(p), mỗi bước thêm 1 ký tự
    pass

# Tự làm xong rồi mới mở test:
# assert find_anagrams("cbaebabacd", "abc") == [0, 6]
# assert find_anagrams("abab", "ab") == [0, 1, 2]

Gợi ý mức 1: bạn có thể gọi thẳng `find_substring_sliding(s, list(p))`! Nhưng hãy thử tự viết bản riêng.
Gợi ý mức 2: ký tự lạ (không có trong `p`) ⇒ xoá sạch, `left = right + 1`.
Gợi ý mức 3: ghi nhận khi `right - left + 1 == len(p)`.

### Bài 6 — Mini challenge: cửa sổ cố định + biến `matched`

Viết lại bài 30 theo kiểu **cửa sổ cố định đúng `m` từ** trên mỗi làn:

- Mỗi bước thêm từ bên phải; nếu cửa sổ đã quá `m` từ thì bỏ từ cách đó `m` bước về bên trái.
- Duy trì `matched` = **số loại từ** `w` có `window[w] == need[w]`.
- Ghi nhận khi `matched == len(need)` (và cửa sổ đủ `m` từ).

Không dùng `while`, không xoá sạch khi gặp từ lạ (từ lạ vẫn được đếm, chỉ là không bao giờ "đủ").

In [27]:
def find_substring_fixed_window(s, words):
    # TODO 1: need = Counter(words); for offset in range(k)
    # TODO 2: window = Counter(); matched = 0
    # TODO 3: thêm từ mới: trước/sau khi tăng, cập nhật matched nếu window[w] chạm/rời need[w]
    # TODO 4: nếu cửa sổ dư 1 từ, bỏ từ bên trái và cập nhật matched tương tự
    # TODO 5: nếu matched == len(need) -> ghi nhận chỉ số bắt đầu cửa sổ
    pass

# Tự làm xong rồi mới mở test:
# assert sorted(find_substring_fixed_window("barfoofoobarthefoobarman", ["bar", "foo", "the"])) == [6, 9, 12]
# assert find_substring_fixed_window("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []

Gợi ý mức 1: `matched` chỉ thay đổi khi `window[w]` **vừa bằng** hoặc **vừa thôi bằng** `need[w]`.
Gợi ý mức 2: khi thêm `w`: nếu `window[w] == need[w]` trước khi tăng thì `matched -= 1`; tăng; nếu giờ `window[w] == need[w]` thì `matched += 1`. Bỏ từ thì làm ngược lại.
Gợi ý mức 3: chỉ cập nhật `window`/`matched` cho các từ **có trong `need`** (bỏ qua từ lạ). Từ rơi khỏi cửa sổ khi thêm `right` là từ tại `right - m*k` (nếu chỉ số đó `>= offset`). Chỉ số bắt đầu cửa sổ là `right - (m-1)*k`.

## 17. Đáp án tham khảo cho bài tập

Nên tự làm trước rồi mới xem phần này.

### Đáp án bài 1

- `"foobarfoobar"`, `["foo","bar"]` ⇒ `total = 6`: các vị trí `0` (`foobar`), `3` (`barfoo`), `6` (`foobar`) ⇒ `[0, 3, 6]`.
- `"abababab"`, `["ab","ba"]` ⇒ `total = 4`: chuỗi con dài 4 chỉ có `"abab"` (vị trí chẵn: cắt thành `ab|ab`, thừa `ab`) và `"baba"` (vị trí lẻ: `ba|ba`) ⇒ `[]`.
- `"aaa"`, `["a","a"]` ⇒ `[0, 1]`.

In [28]:
print(find_substring_sliding("foobarfoobar", ["foo", "bar"]))
print(find_substring_sliding("abababab", ["ab", "ba"]))
print(find_substring_sliding("aaa", ["a", "a"]))
assert find_substring_sliding("foobarfoobar", ["foo", "bar"]) == [0, 3, 6]
assert find_substring_sliding("abababab", ["ab", "ba"]) == []
assert find_substring_sliding("aaa", ["a", "a"]) == [0, 1]

[0, 3, 6]
[]
[0, 1]


### Đáp án bài 2

`need = {aa:1, ab:1}`.

- Làn 0: `aa(0) ab(2) aa(4) ab(6)`
  - `right=0` thêm `aa` ⇒ `{aa:1}`, count 1.
  - `right=2` thêm `ab` ⇒ count 2 ⇒ **khớp tại 0**.
  - `right=4` thêm `aa` ⇒ thừa ⇒ bỏ `aa`@0 ⇒ `left=2` ⇒ count 2 ⇒ **khớp tại 2**.
  - `right=6` thêm `ab` ⇒ thừa ⇒ bỏ `ab`@2 ⇒ `left=4` ⇒ **khớp tại 4**.
- Làn 1: `aa(1) ba(3) aa(5)` ⇒ `ba` là từ lạ ⇒ không có khớp.

Kết quả: `[0, 2, 4]`. Kiểm tra bằng `trace_sliding`:

In [29]:
trace_sliding("aaabaaab", ["aa", "ab"])
assert find_substring_sliding("aaabaaab", ["aa", "ab"]) == [0, 2, 4]

need = {aa:1, ab:1} | m = 2 | k = 2
--- offset 0 ---
right= 0 'aa': left= 0 window={aa:1} count=1
right= 2 'ab': left= 0 window={aa:1, ab:1} count=2  ==> KHỚP tại 0
right= 4 'aa': left= 2 window={aa:1, ab:1} count=2 | thừa 'aa', bỏ ['aa@0']  ==> KHỚP tại 2
right= 6 'ab': left= 4 window={aa:1, ab:1} count=2 | thừa 'ab', bỏ ['ab@2']  ==> KHỚP tại 4
--- offset 1 ---
right= 1 'aa': left= 1 window={aa:1} count=1
right= 3 'ba': từ lạ -> xoá sạch, left=5
right= 5 'aa': left= 5 window={aa:1} count=1
kết quả: [0, 2, 4]


### Đáp án bài 3 → 6

In [30]:
def answer_find_substring_counter_early(s, words):
    k, m = len(words[0]), len(words)
    total = m * k
    need = Counter(words)
    result = []
    for i in range(len(s) - total + 1):
        seen = Counter()
        ok = True
        for j in range(m):
            word = s[i + j * k: i + (j + 1) * k]
            if word not in need:
                ok = False
                break
            seen[word] += 1
            if seen[word] > need[word]:
                ok = False
                break
        if ok:
            result.append(i)
    return result


def answer_my_find_substring(s, words):
    n, m, k = len(s), len(words), len(words[0])
    need = Counter(words)
    result = []
    for offset in range(k):
        left, window, count = offset, Counter(), 0
        for right in range(offset, n - k + 1, k):
            word = s[right:right + k]
            if word not in need:
                window.clear(); count = 0; left = right + k
                continue
            window[word] += 1; count += 1
            while window[word] > need[word]:
                window[s[left:left + k]] -= 1; count -= 1; left += k
            if count == m:
                result.append(left)
    return result


def answer_find_anagrams(s: str, p: str) -> list[int]:
    need = Counter(p)
    window = Counter()
    left = 0
    result = []
    for right, ch in enumerate(s):
        if ch not in need:
            window.clear(); left = right + 1
            continue
        window[ch] += 1
        while window[ch] > need[ch]:
            window[s[left]] -= 1; left += 1
        if right - left + 1 == len(p):
            result.append(left)
    return result


def answer_find_substring_fixed_window(s, words):
    n, m, k = len(s), len(words), len(words[0])
    need = Counter(words)
    result = []
    for offset in range(k):
        window = Counter()
        matched = 0
        for right in range(offset, n - k + 1, k):
            w = s[right:right + k]                      # thêm từ bên phải
            if w in need:
                if window[w] == need[w]:
                    matched -= 1
                window[w] += 1
                if window[w] == need[w]:
                    matched += 1
            start = right - m * k                       # từ vừa rơi khỏi cửa sổ
            if start >= offset:
                lw = s[start:start + k]
                if lw in need:
                    if window[lw] == need[lw]:
                        matched -= 1
                    window[lw] -= 1
                    if window[lw] == need[lw]:
                        matched += 1
            if matched == len(need):
                result.append(right - (m - 1) * k)
    return result


assert answer_find_substring_counter_early("barfoothefoobarman", ["foo", "bar"]) == [0, 9]
assert answer_find_substring_counter_early("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []
assert answer_find_substring_counter_early("barfoofoobarthefoobarman", ["bar", "foo", "the"]) == [6, 9, 12]

assert sorted(answer_my_find_substring("barfoofoobarthefoobarman", ["bar", "foo", "the"])) == [6, 9, 12]
assert sorted(answer_my_find_substring("aaaaaa", ["aa", "aa"])) == [0, 1, 2]
assert answer_my_find_substring("xbarfoo", ["bar", "foo"]) == [1]

assert answer_find_anagrams("cbaebabacd", "abc") == [0, 6]
assert answer_find_anagrams("abab", "ab") == [0, 1, 2]

assert sorted(answer_find_substring_fixed_window("barfoofoobarthefoobarman", ["bar", "foo", "the"])) == [6, 9, 12]
assert answer_find_substring_fixed_window("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []

# Đối chiếu chéo các đáp án với cách 2 trên input ngẫu nhiên
random.seed(438)
for _ in range(300):
    k = random.randint(1, 3)
    m = random.randint(1, 4)
    ws = ["".join(random.choice("ab") for _ in range(k)) for _ in range(m)]
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 18)))
    base = sorted(find_substring_counter(s, ws))
    assert sorted(answer_find_substring_counter_early(s, ws)) == base
    assert sorted(answer_my_find_substring(s, ws)) == base
    assert sorted(answer_find_substring_fixed_window(s, ws)) == base
    p = "".join(random.choice("ab") for _ in range(random.randint(1, 3)))
    assert answer_find_anagrams(s, p) == sorted(find_substring_counter(s, list(p)))

print("Đáp án tham khảo chạy PASS toàn bộ test (kể cả 300 bộ ngẫu nhiên)!")

Đáp án tham khảo chạy PASS toàn bộ test (kể cả 300 bộ ngẫu nhiên)!


## 18. Template cuối cùng nên nhớ

```python
class Solution:
    def findSubstring(self, s: str, words: List[str]) -> List[int]:
        n, m, k = len(s), len(words), len(words[0])
        if n < m * k:
            return []
        need = Counter(words)
        result = []
        for offset in range(k):                         # k làn
            left, window, count = offset, Counter(), 0
            for right in range(offset, n - k + 1, k):   # thêm từ bên phải
                word = s[right:right + k]
                if word not in need:                    # từ lạ -> làm lại
                    window.clear(); count = 0; left = right + k
                    continue
                window[word] += 1; count += 1
                while window[word] > need[word]:        # thừa -> co trái
                    window[s[left:left + k]] -= 1; count -= 1; left += k
                if count == m:                          # đủ -> ghi nhận
                    result.append(left)
        return result
```

Câu thần chú: **"k làn — thêm phải — lạ thì xoá — thừa thì co — đủ m thì ghi"**.



## 19. Checklist tự kiểm tra trước khi submit

- Mình có dùng **bảng đếm** (`Counter`/`dict`) thay vì `set` chưa? (từ lặp trong `words`)
- Mình có lặp **đủ `k` offset** (`for offset in range(k)`) chưa?
- `right` có dừng ở `n - k + 1` để luôn cắt đủ `k` ký tự chưa?
- Gặp từ lạ: đã **xoá sạch** `window`, `count = 0`, và đặt `left = right + k` chưa?
- Khi co cửa sổ: có đủ bộ ba `window[...] -= 1`, `count -= 1`, `left += k` chưa?
- `left` khởi tạo bằng `offset` (không phải `0`) ở mỗi làn chưa?
- Đã thử các ca: ví dụ đề, `s` ngắn hơn `m·k`, `m = 1`, chuỗi toàn một ký tự, đáp án ở offset khác 0 chưa?
- Đề cho trả về **thứ tự bất kỳ** — khi tự test, nhớ `sorted(...)` trước khi so sánh.